# Experimentos TDAH — una corrida

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpospinalo/tdha-revision/blob/main/tdha_experimentos.ipynb)

Ejecuta **una corrida** —un sitio, un subconjunto de ROIs, un enventanado, una
arquitectura— y sube sus resultados. Cada corrida escribe en su propia carpeta (nombre
derivado de la configuración), así que varias personas pueden correr en paralelo sin
chocar; al final se compilan todas juntas.

## Reglas

1. **No cambiar `SEED`.** El 42 mantiene las mismas particiones en todas las corridas, lo
   que hace la comparación pareada.
2. **Editar solo la celda de configuración.** El resto está en `src/`.
3. **No editar a mano los CSV de `results/`.** Repetir una configuración la reemplaza.
4. Si Colab desconecta, reejecutar el notebook: la corrida incompleta se detecta y se rehace.

## Cuánto tarda

Lo domina el subconjunto de ROIs, que fija el tamaño del modelo:

| ROIs | Características | Parámetros | Duración en GPU |
|---|---|---|---|
| 12 | 66 | 100 mil | decenas de minutos |
| 18 | 153 | 145 mil | decenas de minutos |
| 39 | 741 | 446 mil | ~1 hora |
| 116 | 6.670 | 3,5 millones | varias horas |

Para encadenar varias configuraciones en una sesión: `run_queue.py --in-process` (ver
`docs/performance.md`).

## 1. Preparar el entorno

In [ ]:
# Verificar que hay GPU asignada.
# Si no aparece nada: Entorno de ejecución > Cambiar tipo de entorno > GPU
!nvidia-smi -L || echo "SIN GPU — la corrida será mucho más lenta"

In [ ]:
import os, subprocess, sys, importlib.util

REPO = "https://github.com/jpospinalo/tdha-revision.git"
ROOT = "/content/" + REPO.rstrip("/").split("/")[-1].replace(".git", "")

if os.path.exists(ROOT):
    # Actualizar: si la sesión de Colab se reutiliza, el clon puede tener código
    # viejo y la corrida quedaría marcada con un commit que no es el que se ejecutó.
    print(subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=ROOT,
                         capture_output=True, text=True).stdout.strip())
else:
    subprocess.run(["git", "clone", REPO, ROOT], check=True)

# Instalar SOLO lo que falte. Colab ya trae TensorFlow, NumPy, pandas, scikit-learn,
# SciPy, statsmodels, matplotlib y joblib; un "pip install -r requirements.txt" a
# ciegas puede tardar minutos reinstalando TensorFlow y, peor, dejar una versión
# incompatible con el CUDA del entorno.
faltan = [m for m in ["tensorflow", "keras", "sklearn", "numpy", "pandas",
                      "joblib", "scipy", "statsmodels", "matplotlib"]
          if importlib.util.find_spec(m) is None]
if faltan:
    print("instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + faltan, check=True)
else:
    print("todas las dependencias ya están disponibles")

os.chdir(f"{ROOT}/src")
sys.path.insert(0, os.getcwd())
print("listo:", os.getcwd())
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT,
                                capture_output=True, text=True).stdout.strip())

## 2. Configuración

**Esta es la única celda que hay que editar.**

| Parámetro | Valores |
|---|---|
| `SITIO` | `NYU`, `Peking`, `NeuroIMAGE`, `OHSU` |
| `ROI_SET` | `12`, `18`, `39`, `116` |
| `MODELO` | `lstm`, `gru`, `cnn1d`, `transformer`, `deepsets` |

El enventanado se autoselecciona según el sitio (`VENTANA_POR_SITIO`): ventana física de
120 s donde el escaneo lo permite. Para forzar otro, edita `WINDOW_SECONDS`/`STEP_SECONDS`
(o `WINDOW_TR`/`STEP_TR`).

| Sitio | Escaneo | Ventana | Paso | Solape | Nº ventanas |
|---|---|---|---|---|---|
| NYU | 344 s | 120 s (60 TR) | 12 s (6 TR) | 90% | 19 |
| Peking | 464 s | 120 s (60 TR) | 18 s (9 TR) | 85% | 20 |
| NeuroIMAGE | 504 s | 120 s (61 TR) | 20 s (10 TR) | 84% | 20 |
| OHSU | 185 s | — estática — | — | — | 1 |

- **OHSU** (185 s) es demasiado corto para una ventana válida: default estático. `N_SPLITS` a 5.
- **NeuroIMAGE** (39 sujetos): `N_SPLITS` a 5.
- **Peking** (desbalanceado 109/74): activar `CLASS_WEIGHT`.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURACIÓN DE ESTA CORRIDA
# ---------------------------------------------------------------------------
SITIO        = "NYU"
ROI_SET      = "12"
MODELO       = "lstm"
MODEL_ARG    = []        # p. ej. ["units=64", "dropout=0.2"]

# Enventanado por sitio (ventana física de 120 s; respeta el piso de la conectividad
# dinámica ≈ 111 s para el filtrado a 0.009 Hz de ATHENA). Se autoselecciona según SITIO.
VENTANA_POR_SITIO = {
    "NYU":        {"window_seconds": 120, "step_seconds": 12},  # 60/6 TR · ov 90% · ~19 ventanas
    "Peking":     {"window_seconds": 120, "step_seconds": 18},  # 60/9 TR · ov 85% · ~20 ventanas
    "NeuroIMAGE": {"window_seconds": 120, "step_seconds": 20},  # 61/10 TR · ov 84% · ~20 ventanas
    "OHSU":       {"static": True},                             # 185 s: sin ventana válida -> estática
}

_cfg = VENTANA_POR_SITIO[SITIO]
REPRESENTACION = "static" if _cfg.get("static") else "ordered"
WINDOW_SECONDS = _cfg.get("window_seconds")
STEP_SECONDS   = _cfg.get("step_seconds")

# Override avanzado: fija AMBOS en TR para ignorar los segundos (p. ej. 70 y 2 para el
# enventanado histórico). None = usar el default.
WINDOW_TR = None
STEP_TR   = None

N_SPLITS     = 10        # bajar a 5 en NeuroIMAGE y OHSU
N_REPEATS    = 5         # 10 x 5 = 50 evaluaciones externas
CLASS_WEIGHT = False     # activar en Peking
SEED         = 42        # NO CAMBIAR

NOMBRE       = "Juan"        # autor del commit de resultados
CORREO       = "juan@ejemplo.com"
# ---------------------------------------------------------------------------

if REPRESENTACION == "static":
    _vent = "estática (toda la serie)"
elif WINDOW_TR is not None or STEP_TR is not None:
    _vent = f"{WINDOW_TR}/{STEP_TR} TR"
else:
    _vent = f"{WINDOW_SECONDS}/{STEP_SECONDS} s"
print(f"{SITIO} · {ROI_SET} ROIs · ventana {_vent} · {MODELO} "
      f"{MODEL_ARG or ''} · {N_SPLITS}x{N_REPEATS} = {N_SPLITS * N_REPEATS} repeticiones")

# La identidad se fija ANTES de correr: run_experiment.py la registra en config.json.
if NOMBRE and CORREO:
    for k, v in [("user.name", NOMBRE), ("user.email", CORREO)]:
        subprocess.run(["git", "config", k, v], cwd=ROOT, check=True)
    print(f"autor de la corrida: {NOMBRE} <{CORREO}>")
else:
    print("\nAVISO: sin NOMBRE ni CORREO la corrida quedará como 'desconocido' y no se "
          "podrá subir.")


In [ ]:
# Subconjuntos de ROIs y arquitecturas disponibles.
!python run_experiment.py --list-roi-sets
!python run_experiment.py --list-models

## 3. Comprobaciones previas

Dos verificaciones antes de gastar GPU. La primera revisa el repositorio y los datos;
la segunda valida esta configuración concreta y sus particiones sin entrenar.

Conviene mirar el número de ventanas, que no aparezcan avisos, y anotar la `huella`:
dos corridas solo son comparables si coinciden su sitio, su semilla y esa huella.

In [ ]:
!python verify_setup.py

In [ ]:
import io, contextlib, re
import run_experiment


def ejecutar(extra=()):
    """Ejecuta run_experiment en este mismo proceso.

    Arranca TensorFlow una sola vez y comparte las secuencias de conectividad entre la
    prueba de humo y la corrida real. El enventanado se toma de la celda de
    configuración: estático, en segundos, o en TR si se fija el override.
    """
    if REPRESENTACION == "static":
        rep_args = ["--representation", "static"]
        win_args = []
    else:
        rep_args = [] if REPRESENTACION == "ordered" else ["--representation", REPRESENTACION]
        if WINDOW_TR is not None or STEP_TR is not None:
            if WINDOW_TR is None or STEP_TR is None:
                raise ValueError("En modo TR hay que fijar WINDOW_TR y STEP_TR juntos.")
            win_args = ["--window", str(WINDOW_TR), "--step", str(STEP_TR)]
        else:
            win_args = ["--window-seconds", str(WINDOW_SECONDS),
                        "--step-seconds", str(STEP_SECONDS)]

    argv = ["--site", SITIO, "--roi-set", str(ROI_SET), "--model", MODELO,
            *rep_args, *win_args, "--seed", str(SEED),
            "--n-splits", str(N_SPLITS), "--n-repeats", str(N_REPEATS)]
    if MODEL_ARG:
        argv += ["--model-arg"] + list(MODEL_ARG)
    if CLASS_WEIGHT:
        argv += ["--class-weight"]
    argv += list(extra)
    print("run_experiment.py " + " ".join(argv) + "\n", flush=True)

    try:
        return run_experiment.main(argv), None
    except SystemExit as e:
        return None, str(e)


_, aviso = ejecutar(["--dry-run"])
if aviso:
    print(aviso)


## 4. Prueba de humo

Dos pliegues y tres épocas, para verificar que el camino completo funciona antes de
lanzar una corrida larga. Escribe en `/tmp`, así que no ensucia los resultados ni se
sube nada.

In [ ]:
_, aviso = ejecutar(["--n-splits", "2", "--n-repeats", "1",
                     "--epochs", "3", "--patience", "2",
                     "--out", "/tmp/smoke", "--tag", "smoke", "--verbose"])
print("\nprueba de humo:", "FALLÓ\n" + aviso if aviso else "correcta")

## 5. La corrida

Aquí ocurre el trabajo. Si Colab desconecta a mitad, se reejecuta el notebook desde el
principio: la carpeta incompleta se detecta y se rehace.

In [ ]:
RUN_ID, aviso = ejecutar(["--verbose"])

# Repetir una configuración reemplaza la anterior: no quedan dos versiones de lo
# mismo, y una corrida interrumpida se rehace sin más.
if aviso:
    raise RuntimeError(aviso)

print("\n" + "=" * 70)
print("identificador de la corrida:", RUN_ID)

## 6. Resultados

`metrics_val.csv` es el archivo principal: una fila por repetición de la validación
cruzada. Los demás permiten reconstruir después las curvas de convergencia, las
matrices de confusión y las curvas ROC sin reentrenar.

In [ ]:
import pandas as pd
from pathlib import Path

RUTA = Path(ROOT) / "results" / "runs" / RUN_ID
val = pd.read_csv(RUTA / "metrics_val.csv")
tra = pd.read_csv(RUTA / "metrics_train.csv")

print(f"{len(val)} repeticiones · época elegida: mediana {val.best_epoch.median():.0f}, "
      f"rango [{val.best_epoch.min():.0f}, {val.best_epoch.max():.0f}]\n")
M = ["accuracy","balanced_accuracy","f1_macro","precision","recall","specificity","auc"]
resumen = pd.DataFrame({
    "entrenamiento": [tra[m].mean() * 100 for m in M],
    "validación":    [val[m].mean() * 100 for m in M],
    "desv. val.":    [val[m].std() * 100 for m in M],
}, index=M).round(2)
resumen["brecha"] = (resumen.entrenamiento - resumen["validación"]).round(2)
print(resumen.to_string())

La mediana de `best_epoch` muestra cuánto cómputo se gasta tras converger: si queda muy
por debajo del tope de épocas, baja `--patience`. Ver `docs/performance.md`.

In [ ]:
print("archivos generados:\n")
for f in sorted(RUTA.glob("*")):
    filas = f"{sum(1 for _ in f.open()) - 1:>6d} filas" if f.suffix == ".csv" else ""
    print(f"  {f.name:26s} {filas}")

In [ ]:
# La dispersión importa tanto como la media: con muestras pequeñas, repeticiones
# individuales muy altas reflejan particiones favorables, no mejor aprendizaje.
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.hist(val.accuracy * 100, bins=15, edgecolor="white")
a.axvline(val.accuracy.mean() * 100, color="crimson", label="media")
a.set_xlabel("accuracy de validación (%)"); a.set_ylabel("repeticiones"); a.legend()

# Época elegida por pliegue. Si se concentra en valores muy bajos, la partición
# interna es demasiado pequeña para elegir con fiabilidad: ver --start-from-epoch.
b.hist(val.best_epoch, bins=20, edgecolor="white", color="tab:green")
b.axvline(val.best_epoch.median(), color="crimson", label="mediana")
b.set_xlabel("época elegida"); b.set_ylabel("pliegues"); b.legend()
plt.tight_layout(); plt.show()


## 7. Subir los resultados

El push necesita un token de acceso personal de GitHub. **No escribirlo en el
notebook**: se guarda en el panel de secretos de Colab (icono de la llave, a la
izquierda) con el nombre `GITHUB_TOKEN`, y desde ahí se lee sin que quede en el
archivo ni en el historial.

Para crearlo: GitHub → Settings → Developer settings → Personal access tokens →
Fine-grained tokens, con permiso de escritura sobre este repositorio.

In [ ]:
import getpass

if not NOMBRE or not CORREO:
    raise ValueError("Complete NOMBRE y CORREO en la celda de configuración y "
                     "vuelva a ejecutar desde ahí: la identidad debe estar fijada "
                     "ANTES de la corrida para quedar registrada en config.json.")
if not (RUTA / "metrics_val.csv").exists():
    raise FileNotFoundError(f"No hay métricas en {RUTA}. La corrida no terminó.")

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = getpass.getpass("token de GitHub: ")

print("listo para subir:", RUN_ID)

In [ ]:
def git(*args):
    r = subprocess.run(["git"] + list(args), cwd=ROOT, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r.returncode


git("add", f"results/runs/{RUN_ID}")
git("commit", "-m", f"exp: {RUN_ID}")

# Traer primero lo que hayan subido los demás. Como cada corrida vive en su propia
# carpeta, no puede haber conflictos de fusión.
git("pull", "--no-rebase", "origin", "main")

# La URL con el token se arma aquí y no se imprime, para que no quede en la salida
# del notebook ni en la configuración del repositorio.
url = REPO.replace("https://", f"https://{TOKEN}@")
r = subprocess.run(["git", "push", url, "main"], cwd=ROOT,
                   capture_output=True, text=True)
print("push correcto" if r.returncode == 0
      else "FALLÓ el push:\n" + r.stderr.replace(TOKEN, "***"))

## 8. Compilar (cuando ya haya varias corridas)

No hace falta ejecutarlo en cada corrida. Reúne todo lo que haya en `results/runs/`,
lo resume en una tabla y **verifica que las corridas sean comparables**: avisa si
detecta semillas distintas, enventanados distintos, señales BOLD que cambiaron entre
corridas o resultados producidos con código sin confirmar.

In [ ]:
!python compile_results.py

## 9. Diagnóstico de orden (opcional, conviene correrlo primero)

¿El orden temporal de las ventanas aporta señal discriminativa? Corre la **misma**
configuración con tres representaciones: `ordered` (secuencia real), `permuted` (ventanas
barajadas dentro de cada sujeto) y `mean_std` (resumen invariante al orden). Si las tres
rinden parecido, el orden no discrimina, y conviene preferir modelos invariantes al orden
—`deepsets`, o `transformer` con `--model-arg positional=false`— frente a las recurrentes.

Requiere un sitio dinámico (no OHSU). Escribe en `results/` como cualquier corrida, así
que `compile_results.py --stats` las compara luego de forma pareada.

In [ ]:
# Diagnóstico de orden: 3 entrenamientos completos con la config actual. Puede tardar.
# La conectividad ordenada se construye una sola vez y se reutiliza para las tres.
assert REPRESENTACION != "static", "El diagnóstico de orden requiere un sitio dinámico (no OHSU)."
for _rep in ["ordered", "permuted", "mean_std"]:
    print(f"\n=== representación: {_rep} ===")
    _rid, _aviso = ejecutar(["--representation", _rep, "--verbose"])
    if _aviso:
        print(_aviso)